# **Instacart Market Basket Analysis using Apache Spark**

#Environment Setup and Spark Session Initialization

In [ ]:
# Installing required packages
!pip install pyspark
!pip install findspark

In [ ]:
import findspark
findspark.init()

In [ ]:
import pandas as pd
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .appName("Instacart Market Basket Analysis") \
    .getOrCreate()

In [ ]:
spark


# **Loading Dataset**

In [ ]:
import kagglehub

path = kagglehub.dataset_download("psparks/instacart-market-basket-analysis")
print(path)

Using Colab cache for faster access to the 'instacart-market-basket-analysis' dataset.
/kaggle/input/instacart-market-basket-analysis


In [ ]:
import os

products = spark.read.csv(
    os.path.join(path, "products.csv"),
    header=True,
    inferSchema=True,
    quote='"',
    escape='"'
)

aisles = spark.read.csv(
    os.path.join(path, "aisles.csv"),
    header=True,
    inferSchema=True
)

departments = spark.read.csv(
    os.path.join(path, "departments.csv"),
    header=True,
    inferSchema=True
)

orders = spark.read.csv(
    os.path.join(path, "orders.csv"),
    header=True,
    inferSchema=True
)

order_products_prior = spark.read.csv(
    os.path.join(path, "order_products__prior.csv"),
    header=True,
    inferSchema=True
)

order_products_train = spark.read.csv(
    os.path.join(path, "order_products__train.csv"),
    header=True,
    inferSchema=True
)

# **Data Preprocessing**

**Data Type Cleaning**

In [ ]:
from pyspark.sql.functions import col, expr

def to_int(df, col_name):
    # Use try_cast to handle malformed string values by converting them to NULL
    return df.withColumn(col_name, expr(f"try_cast({col_name} as int)"))

orders= to_int(orders, "order_id")
orders= to_int(orders, "user_id")

order_products_prior = to_int(order_products_prior, "order_id")
order_products_prior = to_int(order_products_prior, "product_id")
order_products_prior = to_int(order_products_prior, "add_to_cart_order")
order_products_prior_df = to_int(order_products_prior, "reordered")

order_products_train= to_int(order_products_train, "order_id")
order_products_train= to_int(order_products_train, "product_id")
order_products_train= to_int(order_products_train, "add_to_cart_order")
order_products_train = to_int(order_products_train, "reordered")

products= to_int(products, "product_id")
products_df = to_int(products, "aisle_id")
products= to_int(products, "department_id")

aisles = to_int(aisles, "aisle_id")
departments_df = to_int(departments, "department_id")

**Joining Datasets**

In [ ]:
order_products = order_products_prior.unionByName(order_products_train)

In [ ]:
df = order_products \
    .join(orders, "order_id") \
    .join(products, "product_id") \
    .join(aisles, "aisle_id") \
    .join(departments, "department_id")

In [ ]:
df.printSchema()
df.show(5)
df.count()

root
 |-- department_id: integer (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle: string (nullable = true)
 |-- department: string (nullable = true)

+-------------+--------+----------+--------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+--------------------+----------------+----------+
|department_id|aisle_id|product_id|order_id|add_to_cart_order|reordered|user_id|eval_set|order_number|order_dow|order_hour_of_day|days

33819106

In [ ]:
df.write.mode("overwrite").csv("/content/merged_instacart", header=True)

In [ ]:
sdf = spark.read.csv("/content/merged_instacart", header=True, inferSchema=True)

**Handling Nulls**

In [ ]:
from pyspark.sql.functions import col, sum

sdf.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in sdf.columns
]).show()

+-------------+--------+----------+--------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+------------+-----+----------+
|department_id|aisle_id|product_id|order_id|add_to_cart_order|reordered|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|product_name|aisle|department|
+-------------+--------+----------+--------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+------------+-----+----------+
|            0|       0|         0|       0|                0|        0|      0|       0|           0|        0|                0|               2078068|           0|    0|         0|
+-------------+--------+----------+--------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+------------+-----+----------+



In [ ]:
from pyspark.sql.functions import when

sdf = sdf.withColumn(
    "days_since_prior_order",
    when(col("days_since_prior_order").isNull(), 0)
    .otherwise(col("days_since_prior_order"))
)

In [ ]:
sdf.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in sdf.columns
]).show()

+-------------+--------+----------+--------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+------------+-----+----------+
|department_id|aisle_id|product_id|order_id|add_to_cart_order|reordered|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|product_name|aisle|department|
+-------------+--------+----------+--------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+------------+-----+----------+
|            0|       0|         0|       0|                0|        0|      0|       0|           0|        0|                0|                     0|           0|    0|         0|
+-------------+--------+----------+--------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+------------+-----+----------+



**Handling Duplicates**

In [ ]:
print("Before dropping duplicates:", sdf.count())
sdf = sdf.dropDuplicates()
print("After dropping duplicates:", sdf.count())

Before dropping duplicates: 33819106
After dropping duplicates: 33819106


**Derived Columns**

In [ ]:
from pyspark.sql.functions import count, avg

# Total items per order
items_per_order = df.groupBy("order_id") \
    .agg(count("*").alias("total_items"))

# Reorder ratio
reorder_ratio = df.groupBy("order_id") \
    .agg(avg("reordered").alias("reorder_ratio"))

# Customer order frequency
order_freq = df.select("user_id", "order_id") \
    .distinct() \
    .groupBy("user_id") \
    .agg(count("order_id").alias("order_frequency"))

# Join ONCE
sdf = df \
    .join(items_per_order, "order_id") \
    .join(reorder_ratio, "order_id") \
    .join(order_freq, "user_id")

In [ ]:
sdf.cache()
sdf.count()

33819106

# **Queries**

In [ ]:
sdf.createOrReplaceTempView("instacart")

**Query A**

In [ ]:
spark.sql("""
SELECT department, COUNT(*) AS total_products_ordered
FROM instacart
GROUP BY department
ORDER BY total_products_ordered DESC
LIMIT 10
""").show()

+---------------+----------------------+
|     department|total_products_ordered|
+---------------+----------------------+
|        produce|               9888378|
|     dairy eggs|               5631067|
|         snacks|               3006412|
|      beverages|               2804175|
|         frozen|               2336858|
|         pantry|               1956819|
|         bakery|               1225181|
|   canned goods|               1114857|
|           deli|               1095540|
|dry goods pasta|                905340|
+---------------+----------------------+



The results show that produce, dairy & eggs, and snacks are the most ordered departments, meaning customers mainly buy fresh and essential groceries.

This helps Instacart by guiding inventory planning, restocking priorities, and targeted promotions for high-demand categories.

**Query B**

In [ ]:
spark.sql("""
SELECT department, AVG(reordered) AS reorder_rate
FROM instacart
GROUP BY department
ORDER BY reorder_rate DESC
""").show()

+---------------+-------------------+
|     department|       reorder_rate|
+---------------+-------------------+
|     dairy eggs| 0.6701612678378716|
|      beverages| 0.6536510738452486|
|        produce|  0.650520843762243|
|         bakery| 0.6283806229446914|
|           deli| 0.6081302371433266|
|           pets| 0.6025572044883145|
|         babies| 0.5776798718156188|
|           bulk| 0.5770900590003339|
|         snacks| 0.5744638459399444|
|        alcohol| 0.5712205105025927|
|   meat seafood| 0.5686247189673691|
|      breakfast| 0.5613508346311373|
|         frozen| 0.5426337415452714|
|dry goods pasta| 0.4622197185587735|
|   canned goods| 0.4586390900357624|
|          other|0.40705246022160374|
|      household|0.40333853136634257|
|        missing| 0.3943227040157114|
|  international| 0.3696822037666056|
|         pantry| 0.3474000405760574|
+---------------+-------------------+
only showing top 20 rows


The results show that dairy & eggs, beverages, and produce have the highest reorder rates, meaning they are frequently repurchased items and reflect strong customer loyalty.

This helps Instacart by identifying high-repeat products for subscription offers, personalized recommendations, and ensuring consistent stock availability in these categories

**Query C**

In [ ]:
spark.sql("""
SELECT order_hour_of_day, COUNT(DISTINCT order_id) AS total_orders
FROM instacart
GROUP BY order_hour_of_day
ORDER BY total_orders DESC
""").show()

+-----------------+------------+
|order_hour_of_day|total_orders|
+-----------------+------------+
|               10|      282470|
|               11|      278616|
|               15|      277207|
|               14|      276659|
|               13|      271885|
|               12|      266828|
|               16|      266444|
|                9|      252529|
|               17|      223433|
|               18|      178556|
|                8|      174664|
|               19|      137341|
|               20|      102087|
|                7|       90032|
|               21|       76486|
|               22|       59982|
|               23|       39139|
|                6|       29913|
|                0|       22224|
|                1|       12103|
+-----------------+------------+
only showing top 20 rows


The results show that most orders are placed between 10 AM and 4 PM, with peak activity around 10–11 AM and early afternoon hours. Order volume gradually decreases in the evening and reaches its lowest levels during late night and early morning hours.

This helps Instacart optimize staffing, delivery scheduling, and warehouse operations by focusing resources during peak shopping hours.

**Query D**

In [ ]:
spark.sql("""
SELECT AVG(total_items) AS avg_basket_size
FROM instacart
""").show()

+-----------------+
|  avg_basket_size|
+-----------------+
|15.73547514827861|
+-----------------+



The average basket size is about 15.7 items per order, meaning customers usually buy around 16 products per purchase.

This helps Instacart plan delivery capacity, optimize warehouse operations, and design promotions to increase order size

**Query E**

In [ ]:
spark.sql("""
SELECT product_name, COUNT(*) AS reorder_count
FROM instacart
WHERE reordered = 1
GROUP BY product_name
ORDER BY reorder_count DESC
LIMIT 10
""").show()

+--------------------+-------------+
|        product_name|reorder_count|
+--------------------+-------------+
|              Banana|       415166|
|Bag of Organic Ba...|       329275|
|Organic Strawberries|       214448|
|Organic Baby Spinach|       194939|
|Organic Hass Avocado|       176173|
|     Organic Avocado|       140270|
|  Organic Whole Milk|       118684|
|         Large Lemon|       112178|
| Organic Raspberries|       109688|
|        Strawberries|       104588|
+--------------------+-------------+



The most frequently reordered products are dominated by fresh and organic items such as bananas, organic strawberries, organic spinach, and avocados. These products show extremely high reorder counts, with bananas leading significantly at over 415,000 reorders.

From a business perspective, these items represent everyday essentials and healthy lifestyle preferences, making them highly predictable demand drivers. Instacart should ensure continuous availability of these products and may consider highlighting them in personalized recommendations or subscription-based grocery bundles.

**Query F**

In [ ]:
spark.sql("""
SELECT department,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM instacart), 4) AS percentage_contribution
FROM instacart
GROUP BY department
ORDER BY percentage_contribution DESC
""").show()

+---------------+-----------------------+
|     department|percentage_contribution|
+---------------+-----------------------+
|        produce|                29.2390|
|     dairy eggs|                16.6505|
|         snacks|                 8.8897|
|      beverages|                 8.2917|
|         frozen|                 6.9099|
|         pantry|                 5.7861|
|         bakery|                 3.6227|
|   canned goods|                 3.2965|
|           deli|                 3.2394|
|dry goods pasta|                 2.6770|
|      household|                 2.2906|
|   meat seafood|                 2.1859|
|      breakfast|                 2.1854|
|  personal care|                 1.3859|
|         babies|                 1.2973|
|  international|                 0.8313|
|        alcohol|                 0.4710|
|           pets|                 0.3023|
|        missing|                 0.2289|
|          other|                 0.1126|
+---------------+-----------------

The results show that most orders come from a few key departments, mainly produce (29%), dairy & eggs (16.6%), and snacks. These are everyday essentials that drive customer purchases.

Business insight: Focus inventory and delivery efficiency on these high-demand categories, while using promotions to boost lower-performing departments like alcohol, pets, and others

#User Segmentation (Bonus Task)

In [ ]:
spark.sql("""
WITH user_orders AS (
    SELECT
        user_id,
        COUNT(DISTINCT order_id) AS order_frequency
    FROM instacart
    GROUP BY user_id
)

SELECT
    CASE
        WHEN order_frequency <= 10 THEN 'Low Activity'
        WHEN order_frequency <= 30 THEN 'Medium Activity'
        ELSE 'High Activity'
    END AS user_segment,
    COUNT(*) AS num_users
FROM user_orders
GROUP BY user_segment
ORDER BY num_users DESC
""").show()

+---------------+---------+
|   user_segment|num_users|
+---------------+---------+
|   Low Activity|   107431|
|Medium Activity|    70139|
|  High Activity|    28639|
+---------------+---------+

